# exp03 Arm H — Qwen3-8B-Base QLoRA, head LR 1e-3 / LoRA LR 1e-4, 6 fixed epochs

Generated by `exp03/qlora_armh/build_notebook.py` from the frozen `config.json` and `armh_core.py`. Original competition train rows only (2,285), exp01 split, epoch 6 is the primary checkpoint (fixed in advance). Internet off; bitsandbytes from the `engs108-offline-wheels` dataset. No submission is written.

In [ ]:
import glob, subprocess, sys
wheels = sorted(glob.glob("/kaggle/input/**/bitsandbytes-*.whl", recursive=True))
assert wheels, "attach the engs108-offline-wheels dataset"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", wheels[-1]], check=True)


In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one T4: with 2 visible GPUs the Trainer uses n_gpu=2 for batch/loss scaling

import glob, hashlib, json, math, shutil, time, traceback
import importlib.metadata as metadata

import numpy as np
import pandas as pd
import torch

RUN_START = time.time()
SMOKE_TEST = False  # True only for the tiny pipeline smoke kernel

CONFIG = json.loads(r"""{
 "experiment": "exp03 Arm H: QLoRA with a higher classification-head learning rate, LoRA capacity unchanged",
 "frozen_before_training": true,
 "approval": "User approved Arm H only (GPU), option C: six fixed epochs, epoch 6 is the primary checkpoint; head LR 1e-3, LoRA LR 1e-4",
 "seed": 42,
 "data": {
  "competition": "name-that-llm",
  "file": "train.csv",
  "train_csv_sha256": "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707",
  "external_data": false,
  "split": "exp01: GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42) on prompt_id over train[[id,prompt_id,text,model]] in file order",
  "train_rows": 2285,
  "train_prompts": 127,
  "dev_rows": 414,
  "dev_prompts": 23,
  "training_rows": "all 2285 train rows; the 414 dev rows are never trained on"
 },
 "model": {
  "base": "Kaggle model qwen-lm/qwen-3/Transformers/8b-base/1 (Qwen3-8B-Base)",
  "class": "AutoModelForSequenceClassification(num_labels=18), id2label = sorted model names (exp01 label_map.json)",
  "quantization": {
   "load_in_4bit": true,
   "bnb_4bit_quant_type": "nf4",
   "bnb_4bit_compute_dtype": "float16",
   "bnb_4bit_use_double_quant": true
  },
  "torch_dtype": "float16",
  "device_map": {
   "": 0
  },
  "init": "set_seed(42) immediately before from_pretrained (as exp01), so the score head init equals exp01's",
  "prepare_model_for_kbit_training": {
   "use_gradient_checkpointing": true
  }
 },
 "lora": {
  "task_type": "SEQ_CLS",
  "r": 8,
  "lora_alpha": 16,
  "lora_dropout": 0.05,
  "target_modules": [
   "q_proj",
   "v_proj"
  ],
  "modules_to_save": [
   "score"
  ],
  "bias": "none",
  "expected_trainable_params": {
   "total": 3907584,
   "head": 73728,
   "lora": 3833856
  }
 },
 "training": {
  "max_length": 512,
  "per_device_train_batch_size": 2,
  "gradient_accumulation_steps": 8,
  "effective_batch_size": 16,
  "per_device_eval_batch_size": 8,
  "num_train_epochs": 6,
  "expected_optimizer_steps": 858,
  "steps_per_epoch": 143,
  "optimizer": "Trainer default optim (adamw_torch_fused on torch>=2.8) built via Trainer.get_optimizer_cls_and_kwargs, betas (0.9, 0.999), eps 1e-8, weight_decay 0.0 for both groups (= exp01)",
  "param_groups": {
   "head": {
    "match": "trainable params whose name contains 'score' (modules_to_save copy)",
    "lr": 0.001
   },
   "lora": {
    "match": "trainable params whose name contains 'lora_'",
    "lr": 0.0001
   }
  },
  "lr_scheduler_type": "linear",
  "warmup_steps": 0,
  "schedule_note": "Same scheduler and warmup as exp01 (Trainer defaults: linear decay to 0, no warmup). The decay now spans 6 epochs (858 steps) instead of 3 (429); each group decays from its own peak LR. Steps skipped by the fp16 GradScaler do not advance the schedule (Trainer/accelerate behaviour).",
  "max_grad_norm": 1.0,
  "fp16": true,
  "gradient_checkpointing": true,
  "gradient_checkpointing_kwargs": {
   "use_reentrant": false
  },
  "ga_fix": "trainer.model_accepts_loss_kwargs = False after building the Trainer (the Trainer then divides each micro-batch loss by GA): logged train loss = true mean CE, grad_norm = true norm",
  "eval_strategy": "epoch (diagnostics only; never used to select a checkpoint)",
  "save_strategy": "epoch, save_only_model=True, all 6 epoch checkpoints kept",
  "load_best_model_at_end": false,
  "logging_steps": 10,
  "data_order": "Trainer default RandomSampler seeded by args.seed=42"
 },
 "primary_checkpoint": "end of epoch 6 = optimizer step 858 (the Trainer counts GradScaler-skipped steps in global_step), fixed before training. No early stopping on any metric. Per-epoch dev metrics are diagnostics only.",
 "stop_criteria": {
  "note": "Training stops early ONLY for these predefined criteria. A stopped run is reported as failed/partial; its last state is not substituted as 'epoch 6'.",
  "S1_nonfinite_loss": "mean micro-batch training loss non-finite on 3 consecutive optimizer steps",
  "S2_nonfinite_grad": "pre-clip total gradient norm non-finite (GradScaler skips the step) on 10 consecutive optimizer steps",
  "S3_skip_rate": "cumulative GradScaler skip rate above 5% of optimizer steps, checked from optimizer step 100 onward",
  "S4_operational": "CUDA OOM, Python exception, kernel crash or Kaggle session limit. The same frozen config may be relaunched after fixing an operational bug, documented in the README.",
  "params": {
   "S1_k": 3,
   "S2_k": 10,
   "S3_max_rate": 0.05,
   "S3_min_step": 100
  }
 },
 "logging": {
  "per_optimizer_step": [
   "mean micro-batch train CE (raw, before GA division)",
   "pre-clip grad norm total and per group (head, lora)",
   "post-clip clip coefficient",
   "update norm ||delta theta|| per group",
   "lr per group",
   "GradScaler scale and skipped flag",
   "cosine(head weight, head init)",
   "head weight Frobenius norm"
  ],
  "per_epoch": [
   "dev eval_loss, balanced accuracy, macro F1, log loss (Trainer evaluate, val order, batch 8)",
   "dev logits"
  ],
  "trainer_log_history": true,
  "ga_fix_check": "Trainer-logged loss at each logging step vs the mean of the recorded raw micro-batch losses over the same optimizer steps (should agree to ~1e-4 relative)"
 },
 "post_training_epoch6": {
  "predictions": "length-sorted predict loop of inference/rs2027-engs108-inference.ipynb (batch 8, fp16 autocast, eval mode): dev 414, train 2285, test rows -> logits -> softmax probabilities. These are the primary head predictions.",
  "hidden_states": "exp02 pooled_forward (hook on score input, last non-pad token, batch 8) over train_df then dev_df order (same order as exp02 pooled_hidden_*.npy) -> pooled_hidden_armh.npy (2699 x 4096 float16) + the logits of the same pass; sha256 recorded. Test rows separately.",
  "adapter": "model.save_pretrained(adapter/) + label_map.json",
  "intermediate_layers": "not extracted (proposal only, needs approval)"
 },
 "pre_registered_dev_reports": {
  "note": "Written before training. Dev is reported once for exactly these rows; nothing is selected on dev. Each row gets bal-acc with a 23-prompt bootstrap CI, macro F1, log loss, and paired prompt-bootstrap deltas vs B1 (0.399, exp03/probe/artifacts/blend_B1_dev.csv) and vs R2 probe_exp01 (0.377). stylo_exp03 = exp03/stylo/artifacts/stylo_exp03_dev.csv.",
  "H1": {
   "model": "armh_head",
   "desc": "trained classification head at epoch 6 (length-sorted predict loop), softmax(logits)",
   "weights": "-"
  },
  "H2": {
   "model": "probe_armh",
   "desc": "new linear probe (protocol probe: StandardScaler + LR) on epoch-6 adapted pooled hidden states; C chosen from {0.0003,0.001,0.003,0.01,0.03,0.1} by pooled train-CV OOF bal-acc on the frozen folds (ties within 0.003 -> smallest C). The CV level is optimistic: the adapter saw every train label (oof_honest false).",
   "weights": "-"
  },
  "H3": {
   "model": "probe_armh + stylo_exp03",
   "weights": "equal"
  },
  "H4": {
   "model": "armh_head + stylo_exp03",
   "weights": "equal"
  },
  "H5": {
   "model": "probe_armh + armh_head + stylo_exp03",
   "weights": "equal"
  },
  "H6": {
   "model": "probe_armh + armh_head + stylo_exp03",
   "weights": "dev LOPO (protocol 5b, leave-one-prompt-out over 23 dev prompts)",
   "why_fixed_components": "the 3-component LOPO pool is fixed in advance instead of 'LOPO of the best equal blend', because picking the best equal blend would itself be a dev selection; LOPO can drive any component weight to 0"
  },
  "diagnostics_not_reports": [
   "per-epoch dev metrics from the training log",
   "H1 vs H2 paired delta (does the improved head catch up to the probe?)",
   "train-set fit of the head (in-sample)",
   "pooled_forward logits vs length-sorted logits argmax agreement"
  ]
 },
 "kaggle": {
  "kernel": "ritisingh123/rs2027-engs108-exp03-armh",
  "smoke_kernel": "ritisingh123/rs2027-engs108-exp03-armh-smoke",
  "accelerator": "NvidiaTeslaT4 (CUDA_VISIBLE_DEVICES=0, one T4 used)",
  "docker_image": "gcr.io/kaggle-private-byod/python@sha256:de153e6d303881930c51286216393d56f56b26458d0661350305a1e8144eea01 (same as exp01/exp02 kernels)",
  "internet": false,
  "datasets": [
   "ritisingh123/engs108-offline-wheels (bitsandbytes 0.50.2 wheel)"
  ],
  "expected_runtime_h": 4.0,
  "leaderboard_submission": false
 }
}""")
TC, LC, SC = CONFIG["training"], CONFIG["lora"], CONFIG["stop_criteria"]["params"]
SEED = CONFIG["seed"]
DATA_DIR = "/kaggle/input/competitions/name-that-llm"
MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1"
OUT = "/kaggle/working"
CKPT_DIR = f"{OUT}/checkpoints"
ADAPTER_DIR = f"{OUT}/adapter"
N_CLASSES = 18

versions = {p: metadata.version(p) for p in
            ["torch", "transformers", "peft", "accelerate", "bitsandbytes", "datasets", "scikit-learn", "numpy"]}
print(versions)
print("CUDA:", torch.cuda.is_available(), "| visible GPUs:", torch.cuda.device_count(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


In [ ]:
# ---- armh_core.py (inlined) ----
"""Arm H training utilities (inlined into the Kaggle notebook by build_notebook.py; tested on CPU in tests/).

- `split_param_groups`: head (`score` modules_to_save copy) vs LoRA params, with their own learning rates.
- `ArmHTrainer`: Trainer that records raw micro-batch training losses and per-group pre-clip grad norms.
- `ArmHMonitor`: callback logging, per optimizer step, loss, grad norms, update norms ||delta theta||, lrs,
  GradScaler state, head-vs-init cosine; applies the frozen stop criteria (config.json `stop_criteria`).
- `pooled_forward` (exp02) and `sorted_predict_logits` (exp01 inference notebook) for epoch-6 outputs.
"""
import math

import numpy as np
import torch
from transformers import Trainer, TrainerCallback


def split_param_groups(model):
    head, lora, other = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "score" in n:
            head.append((n, p))
        elif "lora_" in n:
            lora.append((n, p))
        else:
            other.append(n)
    assert not other, f"unexpected trainable params: {other[:5]}"
    assert head and lora
    return head, lora


def build_optimizer(model, training_args, head_lr, lora_lr):
    """Trainer's default optimizer class/kwargs (as exp01), with one param group per block."""
    head, lora = split_param_groups(model)
    opt_cls, opt_kwargs = Trainer.get_optimizer_cls_and_kwargs(training_args, model)
    groups = [{"params": [p for _, p in head], "lr": head_lr, "weight_decay": training_args.weight_decay, "name": "head"},
              {"params": [p for _, p in lora], "lr": lora_lr, "weight_decay": training_args.weight_decay, "name": "lora"}]
    return opt_cls(groups, **opt_kwargs), opt_cls.__name__, {k: v for k, v in opt_kwargs.items() if k != "lr"}


def _norm(params, scale=1.0):
    sq = 0.0
    for p in params:
        if p.grad is not None:
            sq += float(p.grad.detach().float().pow(2).sum())
    return math.sqrt(sq) / scale


class ArmHTrainer(Trainer):
    """Records raw (pre-GA-division) micro-batch losses and per-group pre-clip grad norms."""

    def __init__(self, *a, head_params=None, lora_params=None, **kw):
        super().__init__(*a, **kw)
        self.head_params, self.lora_params = head_params, lora_params
        self.micro_losses, self.last_clip = [], {}

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        out = super().compute_loss(model, inputs, return_outputs=return_outputs, num_items_in_batch=num_items_in_batch)
        if model.training:
            loss = out[0] if return_outputs else out
            self.micro_losses.append(float(loss.detach().float()))
        return out

    def _clip_grad_norm(self, model):
        scaler = getattr(self.accelerator, "scaler", None)
        scale = float(scaler.get_scale()) if scaler is not None and scaler.is_enabled() else 1.0
        head = _norm(self.head_params, scale)
        lora = _norm(self.lora_params, scale)
        total = super()._clip_grad_norm(model)
        self.last_clip = {"grad_norm_head_preclip": head, "grad_norm_lora_preclip": lora,
                          "grad_norm_total_preclip": float(total), "scaler_scale": scale}
        return total


def check_stop(rows, params):
    """Frozen stop criteria on the per-step log rows. Returns a reason string or None."""
    k1, k2 = params["S1_k"], params["S2_k"]
    if len(rows) >= k1 and all(not math.isfinite(r["train_loss"]) for r in rows[-k1:]):
        return f"S1: non-finite train loss on {k1} consecutive optimizer steps"
    if len(rows) >= k2 and all(not math.isfinite(r["grad_norm_total_preclip"]) for r in rows[-k2:]):
        return f"S2: non-finite pre-clip grad norm on {k2} consecutive optimizer steps"
    if len(rows) >= params["S3_min_step"]:
        rate = sum(r["skipped"] for r in rows) / len(rows)
        if rate > params["S3_max_rate"]:
            return f"S3: GradScaler skip rate {rate:.3f} > {params['S3_max_rate']}"
    return None


class ArmHMonitor(TrainerCallback):
    def __init__(self, trainer, stop_params, log_every=10, printer=print):
        self.tr, self.stop_params, self.log_every, self.printer = trainer, stop_params, log_every, printer
        self.head0 = [p.detach().float().clone() for p in trainer.head_params]
        self.rows, self.stop_reason, self._before = [], None, None

    def on_pre_optimizer_step(self, args, state, control, **kw):
        self._before = {"head": [p.detach().float().clone() for p in self.tr.head_params],
                        "lora": [p.detach().float().clone() for p in self.tr.lora_params]}
        self._lrs = {g.get("name", str(i)): g["lr"] for i, g in enumerate(self.tr.optimizer.param_groups)}
        self._post = (_norm(self.tr.head_params), _norm(self.tr.lora_params))

    def on_optimizer_step(self, args, state, control, **kw):
        upd = {}
        for k, params in (("head", self.tr.head_params), ("lora", self.tr.lora_params)):
            sq = sum(float((p.detach().float() - b).pow(2).sum()) for p, b in zip(params, self._before[k]))
            upd[k] = math.sqrt(sq)
        w = torch.cat([p.detach().float().flatten() for p in self.tr.head_params])
        w0 = torch.cat([p.flatten() for p in self.head0])
        losses = self.tr.micro_losses
        self.tr.micro_losses = []
        clip = self.tr.last_clip
        row = {"step": state.global_step + 1, "epoch_float": None,
               "train_loss": float(np.mean(losses)) if losses else float("nan"), "n_micro": len(losses),
               **clip,
               "grad_norm_head_postclip": self._post[0], "grad_norm_lora_postclip": self._post[1],
               "update_norm_head": upd["head"], "update_norm_lora": upd["lora"],
               "lr_head": self._lrs.get("head"), "lr_lora": self._lrs.get("lora"),
               "skipped": bool(self.tr.accelerator.optimizer_step_was_skipped),
               "head_cos_init": float(torch.nn.functional.cosine_similarity(w, w0, dim=0)),
               "head_frob": float(w.norm()), "head_delta_frob": float((w - w0).norm())}
        self.rows.append(row)
        reason = check_stop(self.rows, self.stop_params)
        if reason and self.stop_reason is None:
            self.stop_reason = reason
            control.should_training_stop = True
            self.printer(f"STOP at step {row['step']}: {reason}")
        if row["step"] % self.log_every == 0 or row["step"] == 1:
            self.printer({k: (round(v, 6) if isinstance(v, float) else v) for k, v in row.items()})
        return control

    def on_step_end(self, args, state, control, **kw):
        if self.rows:
            self.rows[-1]["epoch_float"] = state.epoch


def ga_fix_check(rows, log_history):
    """Trainer-logged loss vs mean of recorded raw micro-batch losses over the same optimizer steps."""
    by_step = {r["step"]: r for r in rows}
    out, prev = [], 0
    for h in log_history:
        if "loss" in h and "step" in h and "eval_loss" not in h:
            s = h["step"]
            seg = [by_step[i] for i in range(prev + 1, s + 1) if i in by_step]
            # Trainer weights each step's loss by n_micro/GA; logged = sum(step losses) / n_steps
            ref = float(np.mean([r["train_loss"] for r in seg])) if seg else float("nan")
            out.append({"step": s, "logged": h["loss"], "recorded_mean": ref,
                        "rel_diff": abs(h["loss"] - ref) / max(abs(ref), 1e-12)})
            prev = s
    return out


def pooled_forward(model, score_module, tokenizer, texts, batch_size, max_length, n_classes, device="cuda:0"):
    """exp02 pooled_forward: hidden state at the last non-pad token feeding `score`, plus the model logits."""
    captured = {}
    handle = score_module.register_forward_hook(lambda mod, inp, out: captured.__setitem__("h", inp[0].detach()))
    hidden = np.zeros((len(texts), score_module.in_features if hasattr(score_module, "in_features")
                       else score_module.original_module.in_features), dtype=np.float32)
    logits = np.zeros((len(texts), n_classes), dtype=np.float32)
    try:
        with torch.no_grad():
            for start in range(0, len(texts), batch_size):
                batch = texts[start:start + batch_size]
                enc = tokenizer(batch, truncation=True, max_length=max_length, padding=True, return_tensors="pt").to(device)
                with torch.autocast(device.split(":")[0], dtype=torch.float16, enabled=device.startswith("cuda")):
                    out = model(**enc).logits
                ids = enc["input_ids"]
                non_pad = (ids != tokenizer.pad_token_id).int()
                last = (torch.arange(ids.shape[-1], device=ids.device) * non_pad).argmax(-1)
                h = captured["h"][torch.arange(ids.shape[0], device=ids.device), last]
                hidden[start:start + len(batch)] = h.float().cpu().numpy()
                logits[start:start + len(batch)] = out.float().cpu().numpy()
    finally:
        handle.remove()
    return hidden, logits


@torch.no_grad()
def sorted_predict_logits(model, tokenizer, texts, batch_size, max_length, device="cuda:0"):
    """Length-sorted (longest first) predict loop of inference/rs2027-engs108-inference.ipynb, returning logits."""
    order = np.argsort([len(t) for t in texts])[::-1]
    out = None
    for start in range(0, len(texts), batch_size):
        idx = order[start:start + batch_size]
        enc = tokenizer([texts[i] for i in idx], truncation=True, max_length=max_length, padding=True,
                        return_tensors="pt").to(device)
        with torch.autocast(device.split(":")[0], dtype=torch.float16, enabled=device.startswith("cuda")):
            lg = model(**enc).logits.float().cpu().numpy()
        if out is None:
            out = np.zeros((len(texts), lg.shape[1]), dtype=np.float32)
        out[idx] = lg
    return out


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

assert sha256_file(f"{DATA_DIR}/train.csv") == CONFIG["data"]["train_csv_sha256"]
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
model_names = sorted(train["model"].unique())
label2id = {n: i for i, n in enumerate(model_names)}
id2label = {i: n for n, i in label2id.items()}

df = train[["id", "prompt_id", "text", "model"]].copy()
df["labels"] = df["model"].map(label2id).astype(int)
tr_idx, va_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED).split(df, groups=df["prompt_id"]))
train_df = df.iloc[tr_idx].reset_index(drop=True)
val_df = df.iloc[va_idx].reset_index(drop=True)
DEV_PROMPTS = ['p009', 'p012', 'p018', 'p019', 'p026', 'p029', 'p031', 'p036', 'p055', 'p056', 'p064', 'p068',
               'p069', 'p073', 'p076', 'p078', 'p082', 'p104', 'p110', 'p118', 'p127', 'p132', 'p141']
assert (len(train_df), train_df["prompt_id"].nunique()) == (2285, 127)
assert (len(val_df), val_df["prompt_id"].nunique()) == (414, 23)
assert not set(train_df["prompt_id"]) & set(val_df["prompt_id"])
assert sorted(val_df["prompt_id"].unique()) == DEV_PROMPTS
assert (val_df["model"].value_counts() == 23).all() and val_df["model"].nunique() == N_CLASSES
print("Split verified: 2285/127 train, 414/23 dev (exp01 split)")
full_train_df, full_val_df = train_df, val_df

NUM_EPOCHS, EXPECTED_STEPS = TC["num_train_epochs"], TC["expected_optimizer_steps"]
if SMOKE_TEST:
    train_df = (train_df.assign(n=train_df["text"].str.len()).sort_values("n", ascending=False)
                .head(64).drop(columns="n").reset_index(drop=True))
    val_df = val_df.groupby("labels").head(2).reset_index(drop=True)
    NUM_EPOCHS, EXPECTED_STEPS = 2, 8
    print(f"SMOKE_TEST: train={len(train_df)} val={len(val_df)} epochs={NUM_EPOCHS}")


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                          bnb_4bit_use_double_quant=True)
set_seed(SEED)  # exp01: set_seed immediately before from_pretrained -> same score-head init
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, num_labels=N_CLASSES, id2label=id2label, label2id=label2id, quantization_config=qcfg,
    device_map={"": 0}, torch_dtype=torch.float16, local_files_only=True)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    task_type=TaskType.SEQ_CLS, r=LC["r"], lora_alpha=LC["lora_alpha"], lora_dropout=LC["lora_dropout"],
    target_modules=LC["target_modules"], modules_to_save=LC["modules_to_save"], bias=LC["bias"]))
model.print_trainable_parameters()
head_named, lora_named = split_param_groups(model)
n_head, n_lora = sum(p.numel() for _, p in head_named), sum(p.numel() for _, p in lora_named)
print("head tensors:", [n for n, _ in head_named], "| head params", n_head, "| lora params", n_lora)
exp = LC["expected_trainable_params"]
assert (n_head + n_lora, n_head, n_lora) == (exp["total"], exp["head"], exp["lora"])
score_wrapper = model.base_model.model.score
SCORE_IN = head_named[0][1].shape[1]
np.save(f"{OUT}/score_head_init.npy", head_named[0][1].detach().float().cpu().numpy())


In [ ]:
from datasets import Dataset
from sklearn.metrics import balanced_accuracy_score, f1_score, log_loss
from transformers import DataCollatorWithPadding, TrainingArguments


def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=TC["max_length"])


train_tok = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False).map(tokenize, batched=True, remove_columns=["text"])
val_tok = Dataset.from_pandas(val_df[["text", "labels"]], preserve_index=False).map(tokenize, batched=True, remove_columns=["text"])
epoch_val_logits = []


def softmax_np(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = logits[0] if isinstance(logits, tuple) else logits
    epoch_val_logits.append(np.asarray(logits, dtype=np.float32))
    pred = logits.argmax(-1)
    return {"balanced_accuracy": balanced_accuracy_score(labels, pred),
            "macro_f1": f1_score(labels, pred, average="macro", labels=range(N_CLASSES), zero_division=0),
            "log_loss_proba": log_loss(labels, np.clip(softmax_np(logits.astype(np.float64)), 1e-12, 1), labels=range(N_CLASSES))}


args = TrainingArguments(
    output_dir=CKPT_DIR, per_device_train_batch_size=TC["per_device_train_batch_size"],
    per_device_eval_batch_size=TC["per_device_eval_batch_size"], gradient_accumulation_steps=TC["gradient_accumulation_steps"],
    learning_rate=TC["param_groups"]["lora"]["lr"], num_train_epochs=NUM_EPOCHS, lr_scheduler_type=TC["lr_scheduler_type"],
    warmup_steps=TC["warmup_steps"], max_grad_norm=TC["max_grad_norm"], fp16=True, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}, eval_strategy="epoch", save_strategy="epoch",
    save_total_limit=None, save_only_model=True, load_best_model_at_end=False, logging_steps=TC["logging_steps"],
    label_names=["labels"], report_to="none", seed=SEED)
assert args.weight_decay == 0.0
optimizer, opt_name, opt_kwargs = build_optimizer(model, args, TC["param_groups"]["head"]["lr"], TC["param_groups"]["lora"]["lr"])
print("optimizer:", opt_name, opt_kwargs, [(g["name"], g["lr"], len(g["params"])) for g in optimizer.param_groups])

trainer = ArmHTrainer(model=model, args=args, train_dataset=train_tok, eval_dataset=val_tok,
                      data_collator=DataCollatorWithPadding(tokenizer=tokenizer), compute_metrics=compute_metrics,
                      optimizers=(optimizer, None),
                      head_params=[p for _, p in head_named], lora_params=[p for _, p in lora_named])
trainer.model_accepts_loss_kwargs = False  # GA fix: Trainer divides each micro-batch loss by GA
monitor = ArmHMonitor(trainer, SC, log_every=10)
trainer.add_callback(monitor)
print("n_gpu:", args.n_gpu, "| model_accepts_loss_kwargs:", trainer.model_accepts_loss_kwargs)
assert args.n_gpu == 1


In [ ]:
status, error = "running", None
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
try:
    train_result = trainer.train()
    status = "stopped: " + monitor.stop_reason if monitor.stop_reason else "completed"
except Exception as e:  # S4 operational failure: keep the logs, re-raise after saving
    status, error = f"failed: {type(e).__name__}", traceback.format_exc()
    print(error)
train_minutes = (time.time() - t0) / 60
print("status:", status, "| global_step:", trainer.state.global_step, f"| {train_minutes:.1f} min")
peak = {"allocated_gb": torch.cuda.max_memory_allocated() / 1e9, "reserved_gb": torch.cuda.max_memory_reserved() / 1e9}
print(peak)

step_log = pd.DataFrame(monitor.rows)
step_log.to_csv(f"{OUT}/step_log.csv", index=False)
with open(f"{OUT}/trainer_log_history.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=1)
ga_check = ga_fix_check(monitor.rows, trainer.state.log_history)
pd.DataFrame(ga_check).to_csv(f"{OUT}/ga_fix_check.csv", index=False)
epoch_hist = [h for h in trainer.state.log_history if "eval_balanced_accuracy" in h]
for h in epoch_hist:
    print(f"epoch {h['epoch']:.2f}: eval_loss={h['eval_loss']:.4f} bal_acc={h['eval_balanced_accuracy']:.4f} "
          f"macro_f1={h['eval_macro_f1']:.4f}")
if epoch_val_logits:
    np.save(f"{OUT}/epoch_dev_logits.npy", np.stack(epoch_val_logits))  # (n_epochs, n_dev, 18), val_df order
print("GA-fix check max rel diff:", max([r["rel_diff"] for r in ga_check], default=float("nan")))
print("skipped steps:", int(step_log["skipped"].sum()) if len(step_log) else None)


In [ ]:
completed = status == "completed" and trainer.state.global_step == EXPECTED_STEPS
final_ckpt = f"{CKPT_DIR}/checkpoint-{trainer.state.global_step}"
print("epoch-6 primary checkpoint:", final_ckpt, os.path.isdir(final_ckpt), "| completed:", completed)
model.save_pretrained(ADAPTER_DIR)
with open(f"{ADAPTER_DIR}/label_map.json", "w") as f:
    json.dump({"id2label": {str(i): n for i, n in id2label.items()}}, f, indent=2)

model.eval()
EB, ML = TC["per_device_eval_batch_size"], TC["max_length"]
ev_train, ev_dev = (train_df, val_df) if SMOKE_TEST else (full_train_df, full_val_df)
if SMOKE_TEST:
    ev_train = ev_train.head(40)
t1 = time.time()
logit_cols = [f"logit_{n}" for n in model_names]
parts = []
for name, d in [("train", ev_train), ("dev", ev_dev), ("test", test)]:
    lg = sorted_predict_logits(model, tokenizer, d["text"].tolist(), EB, ML)
    meta = d[["id"]].copy()
    meta["prompt_id"] = d["prompt_id"].values if "prompt_id" in d else None
    meta["labels"] = d["labels"].values if "labels" in d else -1
    meta["split"] = name
    parts.append(pd.concat([meta.reset_index(drop=True), pd.DataFrame(lg, columns=logit_cols)], axis=1))
sorted_logits = pd.concat(parts, ignore_index=True)
sorted_logits.to_parquet(f"{OUT}/armh_sorted_logits.parquet", index=False)
dv = sorted_logits[sorted_logits.split == "dev"]
dev_bal_acc = balanced_accuracy_score(dv["labels"], dv[logit_cols].to_numpy().argmax(1))
print(f"length-sorted predict: dev bal-acc {dev_bal_acc:.4f} [{(time.time() - t1) / 60:.1f} min]")

t2 = time.time()
h_tr, l_tr = pooled_forward(model, score_wrapper, tokenizer, ev_train["text"].tolist(), EB, ML, N_CLASSES)
h_dv, l_dv = pooled_forward(model, score_wrapper, tokenizer, ev_dev["text"].tolist(), EB, ML, N_CLASSES)
h_te, l_te = pooled_forward(model, score_wrapper, tokenizer, test["text"].tolist(), EB, ML, N_CLASSES)
H = np.concatenate([h_tr, h_dv]).astype(np.float16)
np.save(f"{OUT}/pooled_hidden_armh.npy", H)
np.save(f"{OUT}/pooled_hidden_armh_test.npy", h_te.astype(np.float16))
pmeta = pd.concat([ev_train[["id", "prompt_id", "labels"]].assign(split="train"),
                   ev_dev[["id", "prompt_id", "labels"]].assign(split="dev")], ignore_index=True)
pd.concat([pmeta, pd.DataFrame(np.concatenate([l_tr, l_dv]), columns=logit_cols)], axis=1).to_parquet(
    f"{OUT}/armh_pooled_pass_logits.parquet", index=False)
pd.concat([test[["id"]].reset_index(drop=True), pd.DataFrame(l_te, columns=logit_cols)], axis=1).to_parquet(
    f"{OUT}/armh_pooled_pass_test_logits.parquet", index=False)
W = head_named[0][1].detach().float().cpu().numpy()
recon = float(np.abs(np.concatenate([l_tr, l_dv]) - np.concatenate([h_tr, h_dv]) @ W.T).max())
print(f"pooled pass: {(time.time() - t2) / 60:.1f} min | max |logits - H W^T| = {recon:.4f}")


In [ ]:
files = {}
for root, _, fns in os.walk(OUT):
    for fn in fns:
        p = os.path.join(root, fn)
        files[os.path.relpath(p, OUT)] = {"bytes": os.path.getsize(p), "sha256": sha256_file(p)}
with open(f"{OUT}/SHA256SUMS", "w") as f:
    for k in sorted(files):
        f.write(f"{files[k]['sha256']}  {k}\n")
summary = {"status": status, "completed_epoch6": completed, "error": error, "smoke_test": SMOKE_TEST,
           "global_step": trainer.state.global_step, "expected_steps": EXPECTED_STEPS, "stop_reason": monitor.stop_reason,
           "train_minutes": train_minutes, "total_minutes": (time.time() - RUN_START) / 60, "peak_gpu_memory": peak,
           "versions": versions, "optimizer": {"class": opt_name, "kwargs": {k: str(v) for k, v in opt_kwargs.items()}},
           "n_gpu": args.n_gpu, "model_accepts_loss_kwargs": trainer.model_accepts_loss_kwargs,
           "skipped_steps": int(step_log["skipped"].sum()) if len(step_log) else None,
           "ga_fix_check_max_rel_diff": max([r["rel_diff"] for r in ga_check], default=None),
           "epoch_dev_history": epoch_hist, "dev_bal_acc_sorted_predict": dev_bal_acc,
           "pooled_logit_reconstruction_max_abs": recon, "final_checkpoint": final_ckpt,
           "checkpoints": sorted(os.listdir(CKPT_DIR)) if os.path.isdir(CKPT_DIR) else [],
           "config": CONFIG, "files": files}
with open(f"{OUT}/run_summary.json", "w") as f:
    json.dump(summary, f, indent=1, default=str)
print(json.dumps({k: v for k, v in summary.items() if k not in ("config", "files")}, indent=1, default=str))
if error is not None:
    raise RuntimeError(f"training failed operationally: {status}")
